In [1]:
import os
import polars as pl

from typing import Dict, List, Any

from llm_benchmark.utils.dataset import Dataset, DatasetModule
from llm_benchmark.utils import benchmark
from llm_benchmark.data import eval
from llm_benchmark.display import plotting

def tally_single_model(answer_suffix: str
                       ) -> Dict[str, object]:
    dataset: Dataset = benchmark.seshat_setup(seshat_cache_dir="/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/seshat", force=False)
    tally_all: List[Dict[str, Any]] = eval.tally_directory(dataset=dataset, dir=os.path.join("/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/evaluation/", answer_suffix, "wf_answers"))

    data: Dict[str, object] = {}
    for endpoint, data_single in tally_all.items():
        counts, strings = zip(*data_single.values())
        data.update({endpoint.replace("_answers", ""): zip(data_single.keys(), counts)})
    return data

def timeseries_single(endpoint: str
                      ) -> Dict[str, object]:
    data_Qwen_7B: Dict[str, object]    = tally_single_model(answer_suffix="22_04_2026/run_3_Qwen_Qwen-7B-Chat")
    data_Qwen3_8B: Dict[str, object]   = tally_single_model(answer_suffix="14_05_2026/run_9_Qwen_Qwen3-8B"    )
    data_Qwen2_5_7B: Dict[str, object] = tally_single_model(answer_suffix="14_05_2026/run_11_Qwen_Qwen2.5-7B" )

    datasets: Dict[str, object] = [("Qwen-7B", data_Qwen_7B), ("Qwen2.5-7B", data_Qwen2_5_7B), ("Qwen3-8B", data_Qwen3_8B)]

    data: Dict[str, object] = {}

    for model, (dataset) in datasets:
        for ep, data_single in dataset.items():
            if ep == endpoint:
                data.update({model: data_single})
    return data


def plot_single_endpoint(
    endpoint: str, 
                         plot_absolute: 
                         bool = False, plot_relative: bool = True, figsize: tuple = (20, 6)) -> None:
    data_split: Dict[str, object] = timeseries_single(endpoint=endpoint)
    
    if plot_relative:
        plotting.plot_stacked_bar_chart(
            data=data_split,
            figsize=figsize,
            title="War Features Evaluation " + endpoint,
            absolute=False,
            y_limits=[0, 100],
            y_label="percentage cover [%]"
        )
    if plot_absolute:
        plotting.plot_stacked_bar_chart(
            data=data_split,
            figsize=figsize,
            title="War Features Evaluation " + endpoint,
            absolute=True,
            y_limits=[0, 400],
            y_label="n, absolute count [-]"
        )


/Users/apple/miniconda3/envs/llm_benchmark/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
plot_absolute: bool = False
plot_relative: bool = True

figsize: tuple = (10, 5)

plot_single_endpoint(endpoint="wf/long-walls", plot_absolute=plot_absolute, plot_relative=plot_relative, figsize=figsize)
plot_single_endpoint(endpoint="wf/coppers", plot_absolute=plot_absolute, plot_relative=plot_relative, figsize=figsize)
plot_single_endpoint(endpoint="wf/long-walls", plot_absolute=plot_absolute, plot_relative=plot_relative, figsize=figsize)

datapoints = [timeseries_single(endpoint="wf/long-walls"), timeseries_single(endpoint="wf/coppers")]
aggregate: Dict[str, object] = {}

Attempting dataset refresh. False
core/macro-regions https://seshat-db.com/api/core/macro-regions/
Ignoring polity core/macro-regions as per configuration.
core/regions https://seshat-db.com/api/core/regions/
Ignoring polity core/regions as per configuration.
core/ngas https://seshat-db.com/api/core/ngas/
Ignoring polity core/ngas as per configuration.
core/polities https://seshat-db.com/api/core/polities/
Ignoring polity core/polities as per configuration.
core/capitals https://seshat-db.com/api/core/capitals/
Ignoring polity core/capitals as per configuration.
core/nga-polity-relations https://seshat-db.com/api/core/nga-polity-relations/
Ignoring polity core/nga-polity-relations as per configuration.
core/sections https://seshat-db.com/api/core/sections/
Ignoring polity core/sections as per configuration.
core/subsections https://seshat-db.com/api/core/subsections/
Ignoring polity core/subsections as per configuration.
core/variable-hierarchies https://seshat-db.com/api/core/variable

AttributeError: 'DataFrame' object has no attribute 'values'

In [62]:
def tally_single_model(dataset: Dataset,
                       model_filename: str,
                       model: str
                       ) -> pl.DataFrame:
    tally_df: pl.DataFrame = eval.aggregate_entry_per_hierarchy(
        dataset=dataset, 
        dir=os.path.join("/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/evaluation/", 
                         model_filename, 
                         "wf_answers"),
        model=model
        )
    return tally_df


to_tally: Dict[str, str] = {
    "22_04_2026/run_3_Qwen_Qwen-7B-Chat": "Qwen-7B-Chat",
    "14_05_2026/run_9_Qwen_Qwen3-8B": "Qwen3-8B-Chat",
    "14_05_2026/run_11_Qwen_Qwen2.5-7B": "Qwen2.5-7B-Chat",
}

dataset: Dataset = benchmark.seshat_setup(
    seshat_cache_dir="/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/seshat", 
    force=False
    )

dfs: List[pl.DataFrame] = []

for model_filename, model in to_tally.items():
    dfs.append(tally_single_model(
        dataset=dataset, 
        model_filename=model_filename, 
        model=model
        ))
df: pl.DataFrame = pl.concat(dfs)\
    .with_columns(
        (pl.col("model_answer") == pl.col("actual_answer"))
        .cast(pl.Int8)
        .fill_null(0)
        .alias("result")
    )


Attempting dataset refresh. False
core/macro-regions https://seshat-db.com/api/core/macro-regions/
Ignoring polity core/macro-regions as per configuration.
core/regions https://seshat-db.com/api/core/regions/
Ignoring polity core/regions as per configuration.
core/ngas https://seshat-db.com/api/core/ngas/
Ignoring polity core/ngas as per configuration.
core/polities https://seshat-db.com/api/core/polities/
Ignoring polity core/polities as per configuration.
core/capitals https://seshat-db.com/api/core/capitals/
Ignoring polity core/capitals as per configuration.
core/nga-polity-relations https://seshat-db.com/api/core/nga-polity-relations/
Ignoring polity core/nga-polity-relations as per configuration.
core/sections https://seshat-db.com/api/core/sections/
Ignoring polity core/sections as per configuration.
core/subsections https://seshat-db.com/api/core/subsections/
Ignoring polity core/subsections as per configuration.
core/variable-hierarchies https://seshat-db.com/api/core/variable

Qwen2.5-7B-Chat: 100%|██████████| 49/49 [00:13<00:00,  3.59it/s]


In [67]:
def group(
        df: pl.DataFrame,
        metric: str,
        ) -> pl.DataFrame:
    return df\
    .select(["llm_model", "result", metric])\
    .group_by(["llm_model", metric])\
    .agg(
        (pl.col("result").mean() * 100).round(1).alias("%")
    )\
    .sort(["llm_model"])\
    .pivot(
        on="llm_model",
        index=metric,
        values="%"
    )

print(df.columns)

['seshat_entry_id', 'question_entry_id', 'model_answer', 'actual_answer', 'region_idx', 'region_str', 'macro_idx', 'macro_str', 'endpoint', 'llm_model', 'result']


In [68]:
macro_region: pl.DataFrame = group(df=df, metric="macro_str")

with pl.Config(tbl_rows=200, set_tbl_width_chars=100):
    print(macro_region.head(500))

shape: (10, 4)
┌──────────────────────────────┬──────────────┬─────────────────┬───────────────┐
│ macro_str                    ┆ Qwen-7B-Chat ┆ Qwen2.5-7B-Chat ┆ Qwen3-8B-Chat │
│ ---                          ┆ ---          ┆ ---             ┆ ---           │
│ str                          ┆ f64          ┆ f64             ┆ f64           │
╞══════════════════════════════╪══════════════╪═════════════════╪═══════════════╡
│ Southwest Asia               ┆ 26.2         ┆ 0.0             ┆ 0.0           │
│ Southeast Asia               ┆ 24.0         ┆ 0.0             ┆ 0.0           │
│ North America                ┆ 28.4         ┆ 0.0             ┆ 0.0           │
│ South Asia                   ┆ 25.2         ┆ 0.0             ┆ 0.0           │
│ Central and Northern Eurasia ┆ 22.8         ┆ 0.0             ┆ 0.0           │
│ South America and Caribbean  ┆ 20.1         ┆ 0.0             ┆ 0.0           │
│ Oceania-Australia            ┆ 27.0         ┆ 0.0             ┆ 0.0           │
│

In [69]:
region: pl.DataFrame = group(df=df, metric="region_str")

with pl.Config(tbl_rows=200, set_tbl_width_chars=100):
    print(region.head(500))

shape: (34, 4)
┌─────────────────────────┬──────────────┬─────────────────┬───────────────┐
│ region_str              ┆ Qwen-7B-Chat ┆ Qwen2.5-7B-Chat ┆ Qwen3-8B-Chat │
│ ---                     ┆ ---          ┆ ---             ┆ ---           │
│ str                     ┆ f64          ┆ f64             ┆ f64           │
╞═════════════════════════╪══════════════╪═════════════════╪═══════════════╡
│ North China             ┆ 34.2         ┆ 0.0             ┆ 0.0           │
│ East Africa             ┆ 16.7         ┆ 0.0             ┆ 0.0           │
│ Iran                    ┆ 26.3         ┆ 0.0             ┆ 0.0           │
│ Maghreb                 ┆ 8.3          ┆ 0.0             ┆ 0.0           │
│ Maritime Southeast Asia ┆ 25.1         ┆ 0.0             ┆ 0.0           │
│ New Guinea              ┆ 27.9         ┆ 0.0             ┆ 0.0           │
│ Caribbean               ┆ 20.4         ┆ 0.0             ┆ 0.0           │
│ Afghanistan             ┆ 27.1         ┆ 0.0             ┆ 